In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error
import os

df = pd.read_parquet('../data/nifty_features.parquet')
print(f"Loaded {len(df)} rows")
print("Columns:", df.columns.tolist())

features = ['Log_Return_1d', 'Log_Return_5d', 'RSI', 'Volatility']

train_window = 1260
test_window = 252

preds = []
actuals = []
dates = []

print(f"\nRunning walk-forward validation...")
print(f"Train window: {train_window} days | Test window: {test_window} days")

for i in range(train_window, len(df) - test_window, test_window):
    train = df.iloc[:i]
    test = df.iloc[i:i+test_window]
    
    X_train, y_train = train[features], train['Target']
    X_test, y_test = test[features], test['Target']
    
    model = xgb.XGBRegressor(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.05,
        random_state=42
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    
    preds.extend(pred)
    actuals.extend(y_test.values)
    dates.extend(test.index)
    print(f"  Trained up to {train.index[-1].date()}, tested {len(pred)} days")

os.makedirs('../reports', exist_ok=True)
results = pd.DataFrame({
    'Date': dates,
    'Actual': actuals,
    'Pred': preds
})
results.to_csv('../reports/baseline_results.csv', index=False)

rmse = np.sqrt(mean_squared_error(results['Actual'], results['Pred']))
mae = mean_absolute_error(results['Actual'], results['Pred'])
direction_acc = ((results['Pred'] > 0) == (results['Actual'] > 0)).mean()

print(f"\n{'='*50}")
print(f"BASELINE MODEL RESULTS (XGBoost MSE)")
print(f"{'='*50}")
print(f"RMSE:                 {rmse:.6f}")
print(f"MAE:                  {mae:.6f}")
print(f"Directional Accuracy: {direction_acc*100:.2f}%")
print(f"Total predictions:    {len(results)}")
print(f"{'='*50}")
print("\nBaseline results saved to reports/baseline_results.csv")

Loaded 2439 rows
Columns: ['Close', 'High', 'Low', 'Open', 'Volume', 'Target', 'Volatility', 'Log_Return_1d', 'Log_Return_5d', 'RSI']

Running walk-forward validation...
Train window: 1260 days | Test window: 252 days
  Trained up to 2019-03-18, tested 252 days
  Trained up to 2020-03-31, tested 252 days
  Trained up to 2021-04-07, tested 252 days
  Trained up to 2022-04-12, tested 252 days

BASELINE MODEL RESULTS (XGBoost MSE)
RMSE:                 0.013411
MAE:                  0.008731
Directional Accuracy: 54.17%
Total predictions:    1008

Baseline results saved to reports/baseline_results.csv
